# Player and game Elo models (v1)

Earlier training record: player-disjoint splits, XGBoost on behavior + Stockfish features, mean baseline and win-rate ablation, plus a game-level model for single-match inference.

**Data:** classical Lichess dump `lichess_db_standard_rated_2017-02.pgn` (Elo 800–2400). See `docs/data.md`. Colab-first with `MyDrive/src` and `MyDrive/DATA/`.

| Model | Table | Use |
|---|---|---|
| **Primary** | `players.parquet` | stable Elo from many games |
| **Secondary** | `games_split.parquet` | future “paste one PGN” path |

Leakage controls: no names, opponent Elo, rating diffs, or URLs. `n_games` is not a feature. `result_mean` only appears in the ablation.

**Paths:** local `data/`, Colab `/content/data`, or Drive `/content/drive/MyDrive/DATA`.


In [ ]:
# Colab only: mount Drive, then install deps (skip locally)
from pathlib import Path
import sys

IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    import subprocess

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "pyarrow", "pandas", "numpy", "scikit-learn", "scipy", "xgboost", "matplotlib",
    ])
    print("[setup] Drive mounted; dependencies installed")
else:
    print("[setup] local runtime — skipping Drive mount")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA_CANDIDATES = [
    Path("data"),
    Path("/content/data"),
    Path("/content/drive/MyDrive/DATA"),
]
DATA = next(
    (path for path in DATA_CANDIDATES if (path / "players.parquet").exists()),
    DATA_CANDIDATES[0],
)
print(f"[config] DATA -> {DATA.resolve() if DATA.exists() else DATA}")

def show(obj):
    try:
        display(obj)
    except NameError:
        print(obj)

SEED = 42
NON_FEATURES = {
    "game_id", "player_key", "color", "elo", "split", "n_games",
    "result_subject", "result_mean", "eco_family", "first_move",
}
RATING_BINS = [800, 1200, 1400, 1600, 1800, 2000, 2400]
GRID = [
    {"max_depth": 4, "learning_rate": 0.05},
    {"max_depth": 6, "learning_rate": 0.05},
    {"max_depth": 6, "learning_rate": 0.1},
    {"max_depth": 8, "learning_rate": 0.05},
]
print(f"[config] seed={SEED}  rating_bins={RATING_BINS}")
print(f"[config] hyperparameter grid size={len(GRID)}")

## 1. Load data & inspect splits

| Table | Grain | Label |
|---|---|---|
| `players` | 1 row / player | median Elo |
| `games_split` | 1 row / side / game | that game's Elo |

Both inherit the same player-level `split`, so test players never leak into training.

In [ ]:
print("[load] reading parquet files...")
players = pd.read_parquet(DATA / "players.parquet")
games = pd.read_parquet(DATA / "games_split.parquet")
print(f"[load] players rows={len(players):,}  cols={players.shape[1]}")
print(f"[load] games   rows={len(games):,}  cols={games.shape[1]}")

print("\n[split] PLAYER counts:")
player_split_counts = players["split"].value_counts()
print(player_split_counts.to_string())
print("\n[split] GAME counts (inherited from players):")
game_split_counts = games["split"].value_counts()
print(game_split_counts.to_string())

engine_cols = [
    c for c in players.columns
    if c.startswith(("cpl_", "eval_", "mistake_", "blunder_", "best_move_"))
]
print(f"\n[load] Stockfish-derived player columns: {len(engine_cols)}")
if len(engine_cols) == 0:
    raise RuntimeError("No Stockfish columns found. Re-run the pipeline with engine analysis.")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
order = [s for s in ("train", "val", "test") if s in player_split_counts.index]
axes[0].bar(order, [player_split_counts[s] for s in order], color=["#2a6fbb", "#e09f3e", "#9e2a2b"])
axes[0].set_title("Players per split")
axes[0].set_ylabel("Count")
for i, s in enumerate(order):
    axes[0].text(i, player_split_counts[s], str(player_split_counts[s]), ha="center", va="bottom")

for split_name, color in [("train", "#2a6fbb"), ("val", "#e09f3e"), ("test", "#9e2a2b")]:
    subset = players.loc[players["split"] == split_name, "elo"]
    if len(subset):
        axes[1].hist(subset, bins=30, alpha=0.55, label=split_name, color=color)
axes[1].set_title("Elo distribution by split")
axes[1].set_xlabel("Median Elo")
axes[1].legend()

if "n_games" in players.columns:
    axes[2].hist(players["n_games"], bins=30, color="#415a77", edgecolor="white")
    axes[2].set_title("Games per player (not a feature)")
    axes[2].set_xlabel("n_games")
else:
    axes[2].axis("off")

fig.suptitle("Dataset overview — splits should look comparable", y=1.02)
fig.tight_layout()
plt.show()
print("[eda] Overlapping Elo histograms across splits = healthier evaluation.")

## 2. Metrics & training utilities

| Metric | Meaning | Use |
|---|---|---|
| **MAE** | average absolute Elo error | primary score vs baseline |
| **RMSE** | like MAE but punishes large misses | secondary |
| **R²** | variance explained vs predicting the mean | higher is better |
| **Spearman** | rank correlation | does stronger play rank higher? |
| **val MAE** | error on validation only | pick hyperparameters |
| **test MAE** | error on held-out players | report once at the end |

Protocol: fit on **train**, select on **val**, score once on **test**. Baseline Spearman is undefined (constant predictions).

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) > 1 and np.std(y_pred) > 0 and np.std(y_true) > 0:
        rho = float(spearmanr(y_true, y_pred).statistic)
    else:
        rho = np.nan  # undefined for constant predictions (mean baseline)
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
        "spearman": rho,
        "n": int(len(y_true)),
    }


def bin_mae(y_true, y_pred, bins=RATING_BINS):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    labels = pd.cut(y_true, bins=bins, include_lowest=True)
    rows = []
    for label in labels.categories:
        mask = labels == label
        if mask.sum() == 0:
            continue
        rows.append({
            "bin": str(label),
            "n": int(mask.sum()),
            "mae": float(mean_absolute_error(y_true[mask], y_pred[mask])),
        })
    return pd.DataFrame(rows)


def numeric_features(frame, include_result=False):
    banned = set(NON_FEATURES)
    if include_result:
        banned.discard("result_mean")
        banned.discard("result_subject")
    columns = [
        column for column in frame.columns
        if column not in banned and pd.api.types.is_numeric_dtype(frame[column])
    ]
    return [column for column in columns if frame[column].notna().any()]


def split_xy(frame, columns):
    parts = {}
    for name, group in frame.groupby("split"):
        parts[name] = (
            group[columns].to_numpy(dtype=float),
            group["elo"].to_numpy(dtype=float),
        )
    return parts


def fit_xgboost(train_x, train_y, val_x, val_y, params):
    """Fit on TRAIN; monitor VAL for early stopping. Test is never passed here."""
    model = XGBRegressor(
        n_estimators=400,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        tree_method="hist",
        early_stopping_rounds=30,
        random_state=SEED,
        n_jobs=4,
        **params,
    )
    model.fit(
        train_x,
        train_y,
        eval_set=[(train_x, train_y), (val_x, val_y)],
        verbose=False,
    )
    return model


def select_model(train_x, train_y, val_x, val_y, label="model"):
    """Grid search scored on VALIDATION only. Test remains sealed."""
    print(f"\n[{label}] ===== HYPERPARAMETER SEARCH =====")
    print(f"[{label}] TRAIN rows={len(train_y):,}  |  VAL rows={len(val_y):,}  |  TEST sealed")
    print(f"[{label}] Fitting each candidate on TRAIN; ranking by VAL MAE...")
    best = None
    best_mae = np.inf
    best_params = None
    rows = []
    for index, params in enumerate(GRID, start=1):
        print(f"[{label}]   candidate {index}/{len(GRID)}  params={params}  -> fit TRAIN, score VAL")
        model = fit_xgboost(train_x, train_y, val_x, val_y, params)
        mae = float(mean_absolute_error(val_y, model.predict(val_x)))
        best_iter = getattr(model, "best_iteration", None)
        rows.append({**params, "val_mae": mae, "best_iteration": best_iter})
        print(f"[{label}]   candidate {index}/{len(GRID)}  VAL MAE={mae:.2f}  best_iteration={best_iter}")
        if mae < best_mae:
            best = model
            best_mae = mae
            best_params = params
    print(f"[{label}] Selected params={best_params}  VAL MAE={best_mae:.2f}")
    print(f"[{label}] ===== END SEARCH (test still unused) =====\n")
    return best, best_params, float(best_mae), pd.DataFrame(rows)


def print_metrics(tag, metrics):
    print(
        f"[{tag}] n={metrics['n']:,}  "
        f"MAE={metrics['mae']:.2f}  RMSE={metrics['rmse']:.2f}  "
        f"R2={metrics['r2']:.3f}  Spearman={metrics['spearman']}"
    )


def plot_diagnostics(y_true, y_pred, title, importance=None, sample_cap=4000):
    """Calibration scatter, stratified MAE, residuals, optional importances."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    residuals = y_pred - y_true
    bins = bin_mae(y_true, y_pred)

    rng = np.random.default_rng(SEED)
    if len(y_true) > sample_cap:
        idx = rng.choice(len(y_true), size=sample_cap, replace=False)
        plot_true, plot_pred = y_true[idx], y_pred[idx]
    else:
        plot_true, plot_pred = y_true, y_pred

    n_cols = 3 if importance is not None else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(5.4 * n_cols, 4.3))
    if n_cols == 2:
        axes = list(axes)

    lo = float(min(plot_true.min(), plot_pred.min()))
    hi = float(max(plot_true.max(), plot_pred.max()))
    axes[0].scatter(plot_true, plot_pred, s=14, alpha=0.45, edgecolors="none", c="#2a6fbb")
    axes[0].plot([lo, hi], [lo, hi], color="black", lw=1.2, label="perfect calibration")
    axes[0].set_xlabel("True Elo")
    axes[0].set_ylabel("Predicted Elo")
    axes[0].set_title(f"{title}: pred vs true")
    axes[0].legend(loc="upper left", fontsize=8)
    axes[0].set_aspect("equal", adjustable="box")

    axes[1].bar(bins["bin"].astype(str), bins["mae"], color="#415a77")
    axes[1].tick_params(axis="x", rotation=35)
    axes[1].set_ylabel("MAE (Elo)")
    axes[1].set_title("Stratified MAE by true-Elo bin")
    for i, row in bins.iterrows():
        axes[1].text(i, row["mae"], f"n={row['n']}", ha="center", va="bottom", fontsize=8)

    if importance is not None:
        top = importance.head(12).iloc[::-1]
        axes[2].barh(top["feature"], top["importance"], color="#e09f3e")
        axes[2].set_title("Top feature importances")
        axes[2].set_xlabel("Importance")

    fig.suptitle(
        f"{title}  |  residual mean={residuals.mean():.1f}, std={residuals.std():.1f}",
        y=1.02,
        fontsize=11,
    )
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(6.2, 3.6))
    ax.hist(residuals, bins=40, color="#778da9", edgecolor="white")
    ax.axvline(0, color="black", lw=1.2)
    ax.set_xlabel("Predicted - true Elo")
    ax.set_ylabel("Count")
    ax.set_title(f"{title}: residual distribution (want centered near 0)")
    fig.tight_layout()
    plt.show()
    print(f"[plot] {title}: diagonal scatter + mid-bin MAE lowest + residuals ~symmetric.")
    return bins


print("[utils] metric + plotting helpers ready")

## 3. Player-level model (primary)

**Target:** median Elo. **Baseline:** TRAIN mean Elo predicted for every TEST player.
**Fit:** TRAIN. **Select:** VAL MAE. **Report:** TEST once.

Good result: TEST MAE well below baseline, Spearman high, extremes may still be weak.

In [ ]:
print("[player] ===== FEATURE MATRIX =====")
player_cols = numeric_features(players)
player_parts = split_xy(players, player_cols)
for required in ("train", "val", "test"):
    if required not in player_parts or len(player_parts[required][1]) == 0:
        raise RuntimeError(f"player split {required} is empty; extract more games or lower min-games")

x_train, y_train = player_parts["train"]
x_val, y_val = player_parts["val"]
x_test, y_test = player_parts["test"]
print(f"[player] features={len(player_cols)}")
print(f"[player] TRAIN n={len(y_train):,}  mean_elo={y_train.mean():.1f}")
print(f"[player] VAL   n={len(y_val):,}  mean_elo={y_val.mean():.1f}")
print(f"[player] TEST  n={len(y_test):,}  mean_elo={y_test.mean():.1f}  (held out until final score)")

print("\n[player] ===== BASELINE (constant = TRAIN mean), scored on TEST =====")
train_mean_elo = float(y_train.mean())
baseline = np.full_like(y_test, fill_value=train_mean_elo)
player_base_metrics = regression_metrics(y_test, baseline)
print(f"[player] baseline TRAIN mean Elo={train_mean_elo:.1f}")
print_metrics("player/baseline/TEST", player_base_metrics)

player_model, player_params, player_val_mae, player_grid = select_model(
    x_train, y_train, x_val, y_val, label="player"
)
print("[player] grid search results (VAL MAE):")
show(player_grid.round(3))

print("[player] ===== VALIDATION SCORE (selected model) =====")
print_metrics("player/xgb/VAL", regression_metrics(y_val, player_model.predict(x_val)))

print("[player] ===== TEST SCORE (final, once) =====")
player_pred = player_model.predict(x_test)
player_metrics = regression_metrics(y_test, player_pred)
print_metrics("player/xgb/TEST", player_metrics)
print(
    f"[player] TEST MAE lift vs baseline: "
    f"{player_base_metrics['mae'] - player_metrics['mae']:.1f} Elo "
    f"({100 * (1 - player_metrics['mae'] / player_base_metrics['mae']):.1f}% relative reduction)"
)

In [ ]:
print("[player] ===== DIAGNOSTICS ON TEST PREDICTIONS =====")
importance = (
    pd.DataFrame({"feature": player_cols, "importance": player_model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
print("[player] top-20 feature importances:")
show(importance.head(20))

player_bins = plot_diagnostics(y_test, player_pred, "Player / TEST", importance)
print("[player] stratified TEST MAE:")
show(player_bins)

try:
    evals = player_model.evals_result()
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    ax.plot(evals["validation_0"]["rmse"], label="TRAIN RMSE", color="#2a6fbb")
    ax.plot(evals["validation_1"]["rmse"], label="VAL RMSE", color="#9e2a2b")
    ax.set_xlabel("Boosting round")
    ax.set_ylabel("RMSE")
    ax.set_title("Player model: TRAIN vs VAL learning curves")
    ax.legend()
    fig.tight_layout()
    plt.show()
    print("[player] Curves should fall then flatten; large TRAIN<<VAL gap suggests overfitting.")
except Exception as exc:
    print(f"[player] learning curve unavailable: {exc}")

### How to read the player plots

- **Pred vs true:** hug the diagonal; a flat cloud means collapsed to the mean.
- **Stratified MAE:** middle bands usually easiest; thin tails often worse.
- **Residuals:** centered near 0; long one-sided tails = systematic bias.
- **Importances:** Stockfish mistake/blunder/CPL on top is expected.
- **Learning curves:** VAL should track TRAIN; a wide gap = overfit.

## 4. Result ablation (leakage check)

Retrain with `result_mean` (win rate). A large TEST gain means outcomes (opponent strength) are leaking in.

**Pass:** ablation TEST MAE ≈ behavior TEST MAE (or worse).

In [ ]:
print("[ablation] ===== FEATURE MATRIX (+ result_mean) =====")
ablation_cols = numeric_features(players, include_result=True)
ablation_parts = split_xy(players, ablation_cols)
ax_train, ay_train = ablation_parts["train"]
ax_val, ay_val = ablation_parts["val"]
ax_test, ay_test = ablation_parts["test"]
print(f"[ablation] features={len(ablation_cols)} (behavior had {len(player_cols)})")
print(f"[ablation] TRAIN n={len(ay_train):,}  VAL n={len(ay_val):,}  TEST n={len(ay_test):,}")

ablation_model, ablation_params, ablation_val_mae, ablation_grid = select_model(
    ax_train, ay_train, ax_val, ay_val, label="ablation"
)
show(ablation_grid.round(3))

print("[ablation] ===== TEST SCORE =====")
ablation_pred = ablation_model.predict(ax_test)
ablation_metrics = regression_metrics(ay_test, ablation_pred)
print_metrics("ablation/xgb/TEST", ablation_metrics)
print_metrics("behavior/xgb/TEST", player_metrics)
delta = ablation_metrics["mae"] - player_metrics["mae"]
status = "LEAK RISK — win rate helps a lot" if delta < -5 else "OK — no meaningful outcome leak"
print(f"[ablation] MAE(ablation) - MAE(behavior) = {delta:+.2f}  => {status}")

fig, ax = plt.subplots(figsize=(5.5, 3.8))
labels = ["behavior\n(TEST)", "ablation\n(+win rate, TEST)"]
vals = [player_metrics["mae"], ablation_metrics["mae"]]
colors = ["#2a6fbb", "#9e2a2b" if delta < -5 else "#2a9d8f"]
ax.bar(labels, vals, color=colors)
ax.set_ylabel("TEST MAE (Elo)")
ax.set_title("Ablation: does win rate buy accuracy?")
for i, v in enumerate(vals):
    ax.text(i, v, f"{v:.1f}", ha="center", va="bottom")
fig.tight_layout()
plt.show()

## 5. Game-level model (single-match path)

Same player-disjoint splits; predict Elo from **one game side**. Expect higher MAE than player-level.
Categoricals (`eco_family`, `first_move`) are one-hot encoded from **TRAIN only**.

In [ ]:
print("[game] ===== FEATURE MATRIX =====")
game_numeric = [c for c in numeric_features(games) if c != "is_white"]
if "is_white" in games.columns:
    game_numeric = ["is_white", *game_numeric]

def design_games(frame):
    cats_cols = ["eco_family", "first_move"]
    encoder = OneHotEncoder(handle_unknown="ignore", max_categories=30, sparse_output=False)
    train_mask = frame["split"] == "train"
    print(f"[game] fitting OneHotEncoder on TRAIN categoricals only (n={train_mask.sum():,})")
    encoder.fit(frame.loc[train_mask, cats_cols].fillna("missing").astype(str))
    pieces = {}
    for name, group in frame.groupby("split"):
        numeric = group[game_numeric].to_numpy(dtype=float)
        encoded = encoder.transform(group[cats_cols].fillna("missing").astype(str))
        pieces[name] = (
            np.hstack([numeric, encoded]),
            group["elo"].to_numpy(dtype=float),
        )
        print(f"[game] split={name:5s}  rows={len(group):,}  feature_dim={pieces[name][0].shape[1]}")
    return pieces, encoder

game_parts, game_encoder = design_games(games)
gx_train, gy_train = game_parts["train"]
gx_val, gy_val = game_parts["val"]
gx_test, gy_test = game_parts["test"]

print("\n[game] ===== BASELINE (TRAIN mean), scored on TEST =====")
game_train_mean = float(gy_train.mean())
game_baseline = np.full_like(gy_test, fill_value=game_train_mean)
game_base_metrics = regression_metrics(gy_test, game_baseline)
print(f"[game] baseline TRAIN mean Elo={game_train_mean:.1f}")
print_metrics("game/baseline/TEST", game_base_metrics)

game_model, game_params, game_val_mae, game_grid = select_model(
    gx_train, gy_train, gx_val, gy_val, label="game"
)
show(game_grid.round(3))

print("[game] ===== VALIDATION SCORE =====")
print_metrics("game/xgb/VAL", regression_metrics(gy_val, game_model.predict(gx_val)))

print("[game] ===== TEST SCORE (final, once) =====")
game_pred = game_model.predict(gx_test)
game_metrics = regression_metrics(gy_test, game_pred)
print_metrics("game/xgb/TEST", game_metrics)
print(f"[game] TEST MAE lift vs baseline: {game_base_metrics['mae'] - game_metrics['mae']:.1f} Elo")

game_importance = pd.DataFrame({
    "feature": (
        game_numeric
        + [f"cat_{n}" for n in game_encoder.get_feature_names_out(["eco_family", "first_move"])]
    )[: len(game_model.feature_importances_)],
    "importance": game_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("[game] ===== DIAGNOSTICS ON TEST =====")
game_bins = plot_diagnostics(gy_test, game_pred, "Game / TEST", game_importance)
show(game_bins)

## 6. Summary & artifacts

Primary artifact = **player** model. Game model is kept for single-PGN serving. Re-run top-to-bottom after data refreshes; do not cherry-pick TEST metrics.

In [ ]:
print("[summary] ===== FINAL METRICS TABLE =====")
summary = pd.DataFrame([
    {"model": "player baseline", "split_scored": "test", **player_base_metrics},
    {"model": "player xgboost", "split_scored": "test", **player_metrics},
    {"model": "player + win-rate ablation", "split_scored": "test", **ablation_metrics},
    {"model": "game baseline", "split_scored": "test", **game_base_metrics},
    {"model": "game xgboost", "split_scored": "test", **game_metrics},
])
show(summary.round(3))

fig, ax = plt.subplots(figsize=(8.5, 4.2))
plot_df = summary.set_index("model")["mae"]
colors = ["#adb5bd", "#2a6fbb", "#2a9d8f", "#adb5bd", "#e09f3e"]
plot_df.plot(kind="bar", ax=ax, color=colors, rot=25)
ax.set_ylabel("TEST MAE (Elo)")
ax.set_title("Model comparison on held-out TEST")
for i, v in enumerate(plot_df.values):
    ax.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
plt.show()

out_dir = DATA if DATA.exists() else Path(".")
player_model.save_model(out_dir / "player_xgb.json")
game_model.save_model(out_dir / "game_xgb.json")
importance.to_csv(out_dir / "player_feature_importance.csv", index=False)
summary.to_csv(out_dir / "metrics_summary.csv", index=False)
print(f"[summary] saved {out_dir / 'player_xgb.json'}")
print(f"[summary] saved {out_dir / 'game_xgb.json'}")
print(f"[summary] saved {out_dir / 'player_feature_importance.csv'}")
print(f"[summary] saved {out_dir / 'metrics_summary.csv'}")

## 7. Error analysis (player TEST)

Largest absolute residuals — check whether misses concentrate in low `n_games`, extreme Elo, or noisy engine features.

In [ ]:
print("[errors] ===== LARGEST PLAYER TEST RESIDUALS =====")
test_players = players.loc[players["split"] == "test"].copy()
test_players["pred"] = player_pred
test_players["residual"] = test_players["pred"] - test_players["elo"]
test_players["abs_err"] = test_players["residual"].abs()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(test_players["elo"], test_players["residual"], s=22, alpha=0.65, c="#9e2a2b")
axes[0].axhline(0, color="black", lw=1)
axes[0].set_xlabel("True Elo")
axes[0].set_ylabel("Residual (pred - true)")
axes[0].set_title("Residuals vs true Elo (TEST)")

if "n_games" in test_players.columns:
    axes[1].scatter(test_players["n_games"], test_players["abs_err"], s=22, alpha=0.65, c="#415a77")
    axes[1].set_xlabel("n_games")
    axes[1].set_ylabel("|error|")
    axes[1].set_title("|error| vs sample size (TEST)")
else:
    axes[1].axis("off")
fig.tight_layout()
plt.show()

cols = [
    "player_key", "elo", "pred", "residual", "n_games",
    "mistake_rate_mean", "blunder_rate_mean", "cpl_mean_mean",
]
cols = [c for c in cols if c in test_players.columns]
show(test_players.nlargest(15, "abs_err")[cols].round(2))
print("[errors] Done. Prefer more coverage / cleaner engine features over endless retuning.")